# 3. Computer vision: pose from broadcast clips

For a sample of pitches (balanced across pitch types):
1. find the pitch's `play_id` and download its Savant clip
2. keep the first camera shot (the center-field view)
3. run MediaPipe Pose and track the pitcher
4. find release (peak throwing-wrist speed)
5. keep only frames **before** `release - cutoff`, so the model can't see the ball come out

Clips belong to MLB. Use them for personal research only and don't redistribute them.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # so cache/ and reports/ live at the repo root
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pitchscout.workspace import Config, get_pitches, get_clips
from pitchscout import plotting as P

# DEMO=True runs on synthetic data with no network (numbers are NOT real).
DEMO = os.environ.get("PITCHSCOUT_DEMO") == "1"
cfg = Config(pitcher="Mason Miller", start="2024-03-28", end="2025-09-28", n_clips=600, demo=DEMO)
if DEMO:
    print("DEMO MODE: synthetic data, results do not describe a real pitcher")

In [ ]:
df, throws = get_pitches(cfg)
clips = get_clips(cfg, df, throws)   # slow the first time (downloads + pose), cached afterwards
print(len(clips.pose), "usable clips")
df.loc[clips.pose.index, "pitch_type"].value_counts()

## Look at one tracked delivery

In [ ]:
from pitchscout.pose import normalize, JOINTS, BONES
row = clips.pose.index[0]
tr, play_id = clips.tracks[row]
xy = normalize(tr.frames, throws)
fig, ax = plt.subplots(figsize=(10, 2.6))
frames = np.linspace(tr.start, tr.end - 1, 6).astype(int)
for k, t in enumerate(frames):
    for a, b in BONES:
        ia, ib = JOINTS.index(a), JOINTS.index(b)
        ax.plot([xy[t, ia, 0] + 3 * k, xy[t, ib, 0] + 3 * k], [xy[t, ia, 1], xy[t, ib, 1]],
                color=P.SERIES[0] if "r_" in a + b else "#c3c2b7", lw=2.5)
    ax.text(3 * k, 2.6, f"{(t - tr.release) / tr.fps:+.2f}s", ha="center", color=P.INK2)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"{P.name(df.loc[row, 'pitch_type'])}: pitcher skeleton before release (throwing arm in blue)", loc="left")
plt.show()

## Throwing-wrist speed and the cutoff

In [ ]:
w = 16 if throws == "R" else 15
sp = np.linalg.norm(np.diff(tr.frames[:, w, :2], axis=0), axis=1) * tr.fps
t = (np.arange(len(sp)) + 1 - tr.release) / tr.fps
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(t, sp, color=P.SERIES[0], lw=2)
ax.axvspan((tr.start - tr.release) / tr.fps, (tr.end - tr.release) / tr.fps, color="#cde2fb", alpha=.6, label="frames the model sees")
ax.axvline(0, color=P.SERIES[1], lw=2, label="release")
ax.set_xlabel("seconds from release"); ax.set_ylabel("wrist speed"); ax.legend(frameon=False); plt.show()

## Skeleton overlay video (real clips only)

Writes `reports/overlay_<play_id>.mp4` and a PNG filmstrip.

In [ ]:
from pathlib import Path
from pitchscout import viz, video
if not DEMO:
    Path("reports").mkdir(exist_ok=True)
    w = viz.joint_weights({"Glove hand": np.ones(8), "Throwing hand": np.ones(8) * .5}, throws)
    viz.render_overlay(video.CLIPS / f"{play_id}.mp4", tr, w, throws,
                       Path(f"reports/overlay_{play_id}.mp4"), Path(f"reports/overlay_{play_id}.png"))
    from IPython.display import Image; display(Image(f"reports/overlay_{play_id}.png"))